# Ghana-LLM — train on Colab GPU

From-scratch Ghanaian / Black-history LLM. **Code** is pulled from GitHub (public repo); **data + checkpoints** live on *your* Google Drive (the corpus is never in git).

### Do this first
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. On your Mac: `bash scripts/package_data.sh`, then upload `ghana-llm-data.tar.gz` to Google Drive at **`MyDrive/ghana-llm/`** (leave it as the .tar.gz — cell 3 unpacks it).
3. **Runtime → Run all.**

In [ ]:
#@title 1) Check the GPU
import torch, subprocess
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout)
print("CUDA:", torch.cuda.is_available(), "| torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"

In [ ]:
#@title 2) Pull the CODE from GitHub (public repo — no token needed)
import os, shutil
if os.path.isdir("ghana-llm"): shutil.rmtree("ghana-llm")
!git clone --depth 1 https://github.com/Eselase-Noble/ghana-llm.git
%cd ghana-llm
!pip install -q -r requirements-colab.txt
print("code ready")

In [ ]:
#@title 3) Mount Drive, unpack the data bundle, link it in
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ghana-llm'
import os, shutil, subprocess
# first time: unpack ghana-llm-data.tar.gz on Drive -> MyDrive/ghana-llm/corpus/...
if not os.path.isdir(f'{DRIVE}/corpus'):
    assert os.path.exists(f'{DRIVE}/ghana-llm-data.tar.gz'), 'Upload ghana-llm-data.tar.gz to MyDrive/ghana-llm/ first'
    subprocess.run(['tar','xzf',f'{DRIVE}/ghana-llm-data.tar.gz','-C',DRIVE], check=True)
# link the corpus from Drive into the repo tree
if os.path.islink('corpus'): os.remove('corpus')
elif os.path.isdir('corpus'): shutil.rmtree('corpus')
os.symlink(f'{DRIVE}/corpus', 'corpus')
# checkpoints persist on Drive (survive Colab session resets)
CKPT_DIR = f'{DRIVE}/checkpoints/gh_v3'; os.makedirs(CKPT_DIR, exist_ok=True)
assert os.path.exists('corpus/tokenized/gh_v3/train.bin'), 'data not found in bundle'
print('data + checkpoints ready at', DRIVE)

In [ ]:
#@title 4) Train on GPU  (RESUMABLE: re-run after a Colab timeout to continue)
!python src/train/train.py \
    --config configs/gpu.json \
    --dataset gh_v3 \
    --device cuda \
    --ckpt-dir "$CKPT_DIR" \
    --resume

In [ ]:
#@title 5) Sample from the model
PROMPT = "Ghana"  #@param {type:"string"}
!python src/generate/sample.py --dataset gh_v3 --device cuda \
    --ckpt "$CKPT_DIR/ckpt.pt" --prompt "$PROMPT" --tokens 150 --temperature 0.8 --top-k 50

---
## (Optional) Grow the corpus ON Colab, then retrain
Scrapes more data, rebuilds + re-tokenizes a new dataset `gh_v4`. Then set `--dataset gh_v4` in the train/sample cells.

To use a bigger model once the corpus is large, swap `configs/gpu.json` → `configs/base_gpu.json`.

In [ ]:
#@title (Optional) Gather more data + re-tokenize
!python src/scrape/black_history.py
!python src/scrape/books.py --limit-per-query 15
!python src/data/build_corpus.py
!python src/data/prepare.py --dataset gh_v4 --glob '*.txt' --vocab-size 16000 --retrain-tokenizer
print('gh_v4 ready; set --dataset gh_v4 above and re-run train')